# Composable Applications{background-color="black" background-image="images/container.jpg" background-size="100%"}

## Biz: Packaged Business Capabilities (PBCs)

:::: {.columns}

::: {.fragment .column width="50%"}
In 2019 "Innovation Insight for Packaged Business Capabilities and Their Role in the Future Composable Enterprise" [Ref](https://www.scribd.com/document/586814392/19-1211-Gartner-Innovation-Insight-for-Packaged-Biz-Capabilties-PBCs-and-Role-in-Composable-Enterprise-78)

> Software components that represent a well-defined business capability, functionally recognizable as such by a business user. Technically, a PBC is a bounded collection of a data schema and a set of services, APIs and event channels
:::
::: {.fragment .column width="50%"}
![](images/pbc-ecommerce.png){.lightbox}
:::
::::

## Architecture: MACH 
:::: {.columns}

::: {.fragment .column width="50%"}

MACH began as an architectural acronym: Microservices, API-first, Cloud-native, Headless, created to define a modern approach to building digital systems. Each pillar represents a shift away from heavy, inflexible platforms toward modular technology that adapts as the business evolves.

- Microservices: Applications are composed of independent components that deploy, scale, and update without disrupting the whole system.
- API-first: Every feature communicates through well-defined interfaces, enabling products and services to integrate seamlessly.
- Cloud-native: Infrastructure is built for resilience, elasticity, and rapid iteration instead of hardware-bound environments.
- Headless: The front end is decoupled from the back end, letting brands deliver content and commerce across any touchpoint.

<https://machalliance.org/> 
:::

::: {.fragment .column width="50%"}
![](images/mars.webp)

<https://machalliance.org/case-studies/mars>
:::

::::

## Development: Make or buy ?

Once we have defined the application design in terms of capabilities (PBCs) and architecture, it is time to develop.

We have at least 3 different paths to bring these building blocks to life:

- Make code and manage the environment (Containers): You write the application and manage where it runs using Docker, deploying it via Docker Swarm or Kubernetes.

- Make code and abstract the environment (Serverless): You write only the core functions, and the cloud provider handles the deployment and infrastructure (FaaS / AWS Lambda).

- Buy and integrate (SaaS): You do not write the capability at all. You integrate an existing service into your application via APIs (e.g., Stripe for payments).

:::{.callout-note .fragment}
## Spoiler
MCP Servers are becoming the new SaaS for AI integrations!
:::

# Make

## Docker Compose 

<https://docs.docker.com/compose/>

Docker Compose is a tool for defining and running multi-container applications. 
It is the key to unlocking a streamlined and efficient development and deployment experience. 

Compose simplifies the control of your entire application stack, making it easy to manage services, networks, and volumes in a single YAML configuration file. Then, with a single command, you create and start all the services
from your configuration file.

Compose works in all environments - production, staging, development, testing, as
well as CI workflows. It also has commands for managing the whole lifecycle of your application:

 - Start, stop, and rebuild services
 - View the status of running services
 - Stream the log output of running services
 - Run a one-off command on a service

:::{.callout-note .fragment}
Compose can be used do create "composable applications" by assemby different PBC, or to create a single PBC.
:::




#### Compose Model
Example from <https://docs.docker.com/compose/intro/compose-application-model/>

::::{.columns}

::: {.fragment .column .lightbox width="50%"}
![](https://docs.docker.com/compose/images/compose-application.webp)
:::

::: {.fragment .column width="50%"}
```yaml
services:
  frontend:
    image: example/webapp
    ports:
      - "443:8043"
    networks:
      - front-tier
      - back-tier
    configs:
      - httpd-config
    secrets:
      - server-certificate

  backend:
    image: example/database
    volumes:
      - db-data:/etc/data
    networks:
      - back-tier

volumes:
  db-data:
    driver: flocker
    driver_opts:
      size: "10GiB"

configs:
  httpd-config:
    external: true

secrets:
  server-certificate:
    external: true

networks:
  # The presence of these objects is sufficient to define them
  front-tier: {}
  back-tier: {}
```
:::
::::

### Hands-On: Hello PBC
Inspired from <https://docs.docker.com/compose/gettingstarted>

:::{.fragment}
The Business Case: The Classroom Attendance Counter

- The university has installed optical sensors at the doors of our laboratory. 
- We need a system to track in real-time exactly how many people are inside the room.
- And provide tools to visualize in the monitors
:::



## Visual map: from local dev to clustered production

```{mermaid}
flowchart LR
    A[Phase 1\nDocker Compose\nSingle machine] --> B[Phase 2\nDocker Swarm\nCluster orchestration]
    B --> C[Phase 3\nDocker Stack\nCompose file reused]

    A1[Goal: fast local setup] --> A
    B1[Goal: HA + self-healing] --> B
    C1[Goal: same YAML skills] --> C
```

:::{.fragment}
Same application journey: laptop → cluster, without changing the core service model.
:::

## 9) Scale the web service

Compose scaling is useful for local experiments: `docker compose up -d --scale web=3`.

In [ ]:
!cd multicontainers-lab && docker compose up -d --scale web=3 && docker compose ps

Add this healthcheck snippet under `web:` in `docker-compose.yml`:

```yaml
healthcheck:
  test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:5000/health')"]
  interval: 10s
  timeout: 3s
  retries: 3
```

In [ ]:
!cd multicontainers-lab && docker compose down -v && docker ps -a --format 'table {{.Names}}\t{{.Status}}'

## Phase 2 — Docker Swarm (Prod phase)

:::: {.columns}

::: {.fragment .column width="50%"}
- **Concept:** clustering + high availability
- **Roles:** **Manager** (control plane) and **Worker** (run tasks)
- **Transition question:** *What happens if your laptop catches fire?*
- **Answer:** Swarm self-heals and redistributes workload
:::

::: {.fragment .column width="50%"}
```bash
docker swarm init
docker node ls
docker service create --name web nginx
docker service scale web=3
```
:::

::::

## Phase 3 — The Bridge (Compose in Swarm)

:::: {.columns}

::: {.fragment .column width="55%"}
- **Aha moment:** Swarm can deploy the same `docker-compose.yml`
- Add `deploy:` to describe replicas/update strategy
- In Swarm, `build:` is ignored (images must be pre-built/pushed)

```yaml
services:
  web:
    image: multicontainers/web:1.0
    deploy:
      replicas: 3
```
:::

::: {.fragment .column width="45%"}
```bash
docker stack deploy -c docker-compose.yml myapp
docker stack ls
docker stack services myapp
```
:::

::::

## Visual: Stack deployment on Swarm

```{mermaid}
flowchart TB
    M[Manager node] -->|schedules tasks| W1[Worker node 1]
    M -->|schedules tasks| W2[Worker node 2]

    subgraph Stack[Stack: myapp]
      SVC[service: web\nreplicas: 3]
    end

    M --> SVC
    SVC --> T1[task 1]
    SVC --> T2[task 2]
    SVC --> T3[task 3]

    T1 --> W1
    T2 --> W2
    T3 --> W1

    Reg[(Image registry)] --> M
    Reg --> W1
    Reg --> W2
```

:::{.fragment}
In Swarm, images must be pre-built and pullable by every node in the cluster.
:::

## Mini lab (end-to-end)

:::: {.columns}

::: {.column width="60%"}
1. Build locally with Compose (`up -d`)
2. Validate app ↔ Redis communication
3. Add `deploy.replicas: 3`
4. Initialize Swarm
5. Deploy stack and inspect services
6. Scale and observe self-healing
:::

::: {.fragment .column width="40%"}
**Deliverable:**
- Working `docker-compose.yml`
- Stack deployed as `myapp`
- Short screenshot/log evidence
:::

::::

## Recap and command cheat sheet

- **Compose:** `docker compose up -d`, `docker compose logs`, `docker compose down`
- **Swarm:** `docker swarm init`, `docker node ls`, `docker service scale`
- **Stack:** `docker stack deploy -c docker-compose.yml myapp`, `docker stack ls`, `docker stack services myapp`